# SQL Business Metrics Query Design

Reusable SQL business metrics executed from Python using SQLite.

## 1. Database Setup

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path("..")
DB_PATH = ROOT / "database" / "business_metrics.db"
conn = sqlite3.connect(DB_PATH)
print("Connected to:", DB_PATH)

## 2. Load Reusable SQL Queries

In [ ]:
def load_query(query_name):
    path = ROOT / "queries" / f"{query_name}.sql"
    return path.read_text(encoding="utf-8")

mau_query = load_query("monthly_active_users")
revenue_query = load_query("revenue_by_segment")
funnel_query = load_query("conversion_funnel")

## 3. Monthly Active Users

In [ ]:
mau = pd.read_sql_query(mau_query, conn)
mau

## 4. Revenue by Segment

In [ ]:
revenue = pd.read_sql_query(revenue_query, conn)
revenue

## 5. Conversion Funnel

In [ ]:
funnel = pd.read_sql_query(funnel_query, conn)
funnel

## 6. Metric Validation

In [ ]:
assert mau.isnull().sum().sum() == 0
assert revenue.isnull().sum().sum() == 0
assert (revenue["total_revenue"] > 0).all()
assert funnel["conversion_pct"].dropna().between(0, 100).all()
assert (revenue["customer_count"] > 0).all()
print("✓ All metrics validated")

## 7. Data Limitation

The source datasets use different customer ID formats: transactions use numeric IDs 101–104, customers.csv uses IDs 1–3, and customer_segment_data.csv uses IDs C0001–C0150. The queries therefore do not assume these datasets represent the same customer population. The conversion funnel reports exact ID matches only.

In [ ]:
conn.close()